# NB38 - trimmed trailing returns as a ranker: a vault-level screen

The track's luck diagnostics say the incumbent's result is carried by a few cycles and a few
names, and that vaults are admitted on trailing returns that may themselves be a few jumps.
This notebook asks the decision-time question directly, on vaults rather than portfolios: does
a trailing return computed with the vault's best k days REMOVED predict its forward 30-day
Sharpe better than the raw trailing return does? The pre-registered workflow decision: if the
vault-level screen does not clear, the idea is not advanced to a portfolio test. That is a
stopping rule, not a claim that a portfolio effect is impossible.

**Focus is forward Sharpe**, not forward return: the operator wants steady profit, and a
trimmed score is expected to cost CAGR. Forward return, volatility and drawdown are reported
beside it. **Verdict: DIAGNOSTIC. No trimmed score is shown to predict forward Sharpe better
than its raw form under a family-wise bound; the trimmed return score's gain, where it appears,
comes with a strong loading on lower forward volatility.**

**The panel is the archive, not the engine's candidate pool.** Every Hypercore vault with at
least the trailing window of history, a TVL of at least 7,500 USD and five price-changing marks
in the window is a candidate on every second day from 2026-04-01 (the polling-density break) to
the last date with a complete 30-day forward window: 17,197 candidate-dates, 364 vaults,
70 decisions to 2026-08-17, 79.9% of rows under 360 days old (cell 4). Eligibility is
on OBSERVED marks: a candidate needs a real mark within 3 days of T-1 (7,705 candidate-dates
dropped for having none, 13,244 for TVL) and a forward window needs at least 10 observed marks
with one in its last 3 days (18 dropped); the median forward window has
30 of 30 days marked. Stratwise Multi-Asset Public (61 days old) and the
other post-July vaults are too young for any forward outcome and are NOT in the screen; they
are shown in a current-snapshot comparison (cell 12). No vault is selected, masked or tuned by name anywhere. Snapshot
`vault-prices.parquet` 255,548,076 bytes, sha256 `11e7c5e0103e1012`, last mark 2026-09-16 (cell 2).

**Based on:** [28-research-stability-signal-screen.ipynb](28-research-stability-signal-screen.ipynb)
and [34-research-calm-score-screen.ipynb](34-research-calm-score-screen.ipynb) for the
two-way cluster bootstrap and simultaneous bounds, re-implemented here without the engine.

## Method

Signals, read at T-1 over trailing windows of 45, 90 and 180 rows: annualised log return (raw
and with the best 3, 5 and 10 daily log returns removed), annualised Sharpe of daily log returns
(raw and trimmed the same way), Sortino, realised volatility. Direction 'high' for return and
Sharpe scores (higher is better), 'low' for volatility. Targets over (T, T + 30 d]: forward
Sharpe (primary), forward log return, forward volatility, forward max drawdown.

Inference: per date, Spearman across that date's candidates, signed so positive means "the
signal's good end had the better outcome", averaged over dates; one two-way cluster bootstrap
(15-decision circular date blocks x vault clusters, 500 draws, seed
20260916) shared across every hypothesis; studentised max-T simultaneous lower bounds
over the family of 30 signals on the primary target (critical value 2.53). The
PAIRED difference trimmed-minus-raw on the primary target, per (window, k), is computed on the
same draws and controlled as its own family of 18 (critical 2.39); per-comparison
intervals and add-one p-values are descriptive. A noisy foresight oracle through the identical
machinery clears the family-wise bound (lower bound 0.989, cell 8), so an all-fail
result is a property of the signals, not of the screen.

## Key new insights and what did we learn from this experiment?

**1. Nothing predicts a vault's next-30-day Sharpe well, and a raw trailing return does not
predict it at all.** The best of thirty signals is the raw 180-day Sharpe at rho
0.136 (unadjusted add-one p 0.016); no signal clears the simultaneous
lower bound of zero over the family (best -0.004, `sharpe180_k5`), while the foresight
oracle clears it at 0.989 (cells 6, 8). Raw trailing return over 45 or 90 days is at
-0.004 and -0.019 - nothing - and every signal's correlation with forward
RETURN is within 0.089 of zero (cell 6). The incumbent's 45-day
Sharpe leg sits at 0.049.

**2. Trimming the return score raises its forward-Sharpe correlation, but not by enough to
establish under a family-wise bound, and the gain arrives with a strong low-volatility
loading.** Removing the best 10 of 90 days lifts the return score from -0.019 to
0.086; the paired difference is 0.106
[0.004, 0.206], add-one p
0.040 on its own, but its simultaneous lower bound over the 18 paired
comparisons is -0.016; at 45 days the difference is
0.100 [-0.007, 0.215] (cell 6). What
the trimmed score correlates with is forward VOLATILITY (signed 0.630 against the raw
score's 0.100) and forward drawdown (0.568), a profile close
to `vol90`'s own (0.676 / 0.611, and 0.095 on
forward Sharpe), and its correlation with forward return stays at 0.004. At the current
snapshot the raw and k = 10 trimmed 90-day return scores rank the cross-section with a Spearman
agreement of 0.122 (cell 12): trimming ten of ninety days re-orders the candidates almost
completely. The result is CONSISTENT with the trimmed score being a stability signal rather than
a cleaner return signal; this notebook does not show that it selects the same vaults as a
volatility ranker, and it makes no portfolio claim.

**3. Trimming the Sharpe score shows no detectable improvement.** Raw and trimmed trailing
Sharpe rank the cross-section similarly (agreement 0.847 at k = 5 on 90 days,
0.949 at k = 3 on 180) and their paired differences on forward Sharpe are
-0.003 [-0.051, 0.036] and
-0.003 [-0.046, 0.032] (cell 6): intervals
that straddle zero, which is absence of evidence of a difference, not evidence of none.

**4. The young cohort shows the largest return-trim differences, and they still do not clear
the family bound.** Among vaults under 360 days (79.9% of rows), raw 90-day return is
-0.038 on forward Sharpe and trimmed k = 10 is 0.088, difference
0.127 [0.016, 0.227], add-one p
0.016, simultaneous lower bound -0.015 over the cohort's
18 comparisons; among the old (3,455 rows, 40 dates) every difference is inside
its interval (cell 10). The young and old screens are separate bootstraps on separate samples.

**5. Stratwise, at the snapshot (2026-09-15, the last completed UTC day).** At the 45-day
window Stratwise's raw return score is 30.9% annualised (Sharpe score 6.44, realised
vol 4.8%); with its best 5 days removed 7.4% and with 10 removed -1.5%, so about
76% of its 45-day return score is its best five days (cell 12). On the raw return
score it ranks 119 of 221 scorable vaults and on the k = 5 trimmed score 20; on the Sharpe score
13 raw and 16 trimmed (cell 12). Those are snapshot ranks on one window and say nothing about
its forward behaviour: with 61 days of history no decision gives it both a trailing
window and a complete forward window.

## Summary of results

Forward-Sharpe screen, all candidates (cell 6): signed Spearman, simultaneous lower bound over
30 signals, unadjusted add-one p; forward volatility column is signed so positive = the score's
good end had LOWER forward volatility.

| signal | rho fwd Sharpe | lower bound | p | rho fwd return | rho fwd vol (signed) |
|---|---|---|---|---|---|
| ret45 raw / k3 / k5 / k10 | -0.004 / 0.042 / 0.068 / 0.095 | -0.015 (k10) | 0.016 | 0.020 | 0.074 -> 0.696 |
| ret90 raw / k3 / k5 / k10 | -0.019 / 0.046 / 0.065 / 0.086 | -0.058 (k10) | 0.066 | 0.004 | 0.100 -> 0.630 |
| ret180 raw / k10 | 0.093 / 0.088 | -0.056 | 0.070 | 0.024 | 0.238 -> 0.566 |
| sharpe45 raw / k5 | 0.049 / 0.048 | -0.091 | 0.178 | -0.035 | 0.127 |
| sharpe90 raw / k5 | 0.032 / 0.028 | -0.102 | 0.285 | -0.066 | 0.122 |
| **sharpe180 raw / k5** | **0.136 / 0.132** | -0.013 | 0.016 | 0.034 | 0.228 |
| sortino 45 / 90 / 180 | 0.043 / 0.023 / 0.132 | -0.018 (180) | 0.016 | 0.033 | 0.252 |
| vol 45 / 90 / 180 (low is good) | 0.099 / 0.095 / 0.059 | -0.008 (45) | 0.010 | 0.042 | 0.733 |

Paired trimmed-minus-raw on forward Sharpe, per-comparison 95% intervals (cell 6; young cohort
cell 10). Simultaneous lower bounds over each 18-comparison family are all below zero.

| | k = 3 | k = 5 | k = 10 |
|---|---|---|---|
| return, 45 d | 0.046 [-0.016, 0.112] | 0.072 [-0.013, 0.163] | 0.100 [-0.007, 0.215] |
| return, 90 d | 0.066 [0.002, 0.127] | 0.084 [0.000, 0.162] | 0.106 [0.004, 0.206] |
| return, 180 d | 0.000 [-0.054, 0.050] | -0.003 [-0.066, 0.060] | -0.005 [-0.085, 0.070] |
| Sharpe, 90 d | -0.009 [-0.038, 0.017] | -0.003 [-0.051, 0.036] | 0.008 [-0.069, 0.073] |
| Sharpe, 180 d | -0.003 [-0.035, 0.022] | -0.003 [-0.046, 0.032] | -0.011 [-0.082, 0.050] |
| return, 90 d, young only | 0.078 [0.010, 0.139] | 0.100 [0.013, 0.178] | 0.127 [0.016, 0.227] |

**What this means for the ranker.** The screen does not establish that a trimmed return score
predicts forward Sharpe better than the raw one, and where it looks better the score has taken
on a strong low-volatility loading, so a trimmed-CAGR leg would be closer to a second stability
leg than to a cleaner return leg. The idea stops at the vault level, as the plan for it said it
should if the screen did not clear. The largest observed forward-Sharpe association is the raw
180-day Sharpe; several others are positive on their own (`vol45` at 0.099, the trimmed
return scores), and none is separable from zero under the family-wise bound. Portfolio
consequences are not claimed here.

## Robustness of results

- The panel is built from the archive with a fixed rule (TVL, history, fresh marks, a recent
  observed mark) and no engine; it is therefore NOT the incumbent's candidate pool (no inclusion
  criteria, quarantine or momentum gate), and its per-date pools are larger
  (~246 candidates). The question asked is about vaults, so that is the right
  population; portfolio consequences are not claimed.
- Eligibility is on observed marks and forward outcomes are coverage-qualified, forward-filled
  daily outcomes (cell 4): no candidate is admitted on a forward-filled TVL or a stale price;
  every forward window has at least 10 real marks and one in its last 3 days (median
  30 marked days of 30); inside a window a day without a mark is still
  forward-filled to a zero return, as in the trailing series.
- The screen is shown reachable: a noisy foresight oracle clears the 31-signal family-wise
  bound at 0.989 (cell 8).
- Every hypothesis shares one bootstrap; paired differences are differences of the same draws,
  so their intervals are paired intervals, and the 18 paired comparisons carry their own
  simultaneous bound. Per-comparison p-values are add-one corrected and descriptive.
- 70 decisions two days apart over about 138 calendar days, each with a 30-day forward window,
  hold roughly four to five non-overlapping forward horizons; the date-block bootstrap uses
  15-decision blocks, and the intervals reflect that overlap.
- Forward max drawdown is in log units (`fwd_log_max_dd`); ranks are unaffected. Trimmed
  scores are ranking transformations, not investable returns.
- Stratwise's figures are a current snapshot at one window on the last completed UTC day and
  are not evidence about its forward behaviour; the rule against selecting or tuning by name
  is unchanged.


## Part 0. Archive, provenance, constants


In [1]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

ARCHIVE = Path.home() / ".cache/tradingstrategy/vaults/downloads/vault-prices.parquet"
raw_bytes = ARCHIVE.read_bytes()
PROVENANCE = {"file": str(ARCHIVE), "bytes": len(raw_bytes), "sha256": hashlib.sha256(raw_bytes).hexdigest()}
del raw_bytes
HYPERCORE_CHAIN = 9999
STRATWISE = "0x0ff219ac20596b457558341bc410bc7a08a1394c"   # display only; never used to select or tune

WINDOWS = (45, 90, 180)
TRIMS = (0, 3, 5, 10)
FORWARD_DAYS = 30
POST_BREAK_START = pd.Timestamp("2026-04-01")
DECISION_STEP_DAYS = 2
MIN_TVL_USD = 7_500.0
MIN_FRESH = 5
#: A candidate must have an ACTUAL mark within this many days before the decision (no stale
#: forward-filled eligibility), and a forward window must contain at least MIN_FORWARD_MARKS
#: observed marks with one in its last MAX_STALE_DAYS days, so a vault that stopped reporting
#: cannot supply manufactured zero-return days as an outcome.
MAX_STALE_DAYS = 3
MIN_FORWARD_MARKS = 10
MIN_CANDIDATES = 8
MIN_DATES = 40
YOUNG_DAYS = 360          # the incumbent's CAGR leg cannot score a vault younger than this
DRAWS = 500
DATE_BLOCK = 15
SEED = 20260916
LEVEL = 0.95

df = pd.read_parquet(ARCHIVE, columns=["address", "chain", "share_price", "total_assets", "name"])
df = df[df["chain"] == HYPERCORE_CHAIN].reset_index()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df[df["timestamp"] >= pd.Timestamp("2025-06-01")].sort_values(["address", "timestamp"])
df["date"] = df["timestamp"].dt.floor("D")
daily = df.groupby(["address", "date"])[["share_price", "total_assets"]].last().reset_index()
names = df.groupby("address")["name"].last()
LAST_MARK = df["timestamp"].max()
display(pd.Series({**PROVENANCE, "last_mark": str(LAST_MARK), "hypercore_vaults": daily["address"].nunique(),
                   "daily_rows": len(daily)}, name="value").to_frame())


,value
file,/Users/moo/.cache/tradingstrategy/vaults/downl...
bytes,255548076
sha256,11e7c5e0103e10125a27fcb2ed58febd394cab3e317771...
last_mark,2026-09-16 07:25:06.705000
hypercore_vaults,604
daily_rows,111006


## Part 1. The panel

One row per (decision date, vault). Trailing series are forward-filled daily marks (a day
without a mark repeats the last one, a zero log return); trimming removes the k LARGEST daily
log returns inside the window before annualising, so a vault whose window return is three jumps
loses most of it. Forward series are the same daily grid over the next 30 days.


In [2]:
def trailing_scores(r: np.ndarray) -> dict:
    """Raw and trimmed annualised log return and Sharpe, Sortino and volatility of one window."""
    out = {}
    n = len(r)
    order = np.sort(r)
    for k in TRIMS:
        kept = order[:n - k] if k else order
        ann = 365.0 / n
        out[f"ret_k{k}"] = float(kept.sum() * ann)
        sd = float(kept.std(ddof=1)) if len(kept) > 2 else float("nan")
        out[f"sharpe_k{k}"] = float(kept.mean() / sd * math.sqrt(365.0)) if sd and sd > 0 else float("nan")
    downside = np.sqrt(np.mean(np.clip(r, None, 0.0) ** 2))
    out["sortino"] = float(r.mean() / downside * math.sqrt(365.0)) if downside > 0 else float("nan")
    out["vol"] = float(r.std(ddof=1) * math.sqrt(365.0))
    return out


def forward_targets(r: np.ndarray, prices: np.ndarray) -> dict:
    sd = float(r.std(ddof=1)) if len(r) > 2 else float("nan")
    path = np.concatenate([[0.0], np.cumsum(r)])
    return {"fwd_return": float(r.sum()),
            "fwd_sharpe": float(r.mean() / sd * math.sqrt(365.0)) if sd and sd > 0 else float("nan"),
            "fwd_vol": float(sd * math.sqrt(365.0)) if sd == sd else float("nan"),
            # Log-unit drawdown (minimum of cumulative log return below its running maximum);
            # ranks are the same as for the percentage form.
            "fwd_log_max_dd": float(np.min(path - np.maximum.accumulate(path)))}


last_decision = (LAST_MARK.floor("D") - pd.Timedelta(days=FORWARD_DAYS))
decisions = pd.date_range(POST_BREAK_START, last_decision, freq=f"{DECISION_STEP_DAYS}D")
rows = []
dropped = {"no_recent_mark": 0, "tvl": 0, "forward_marks": 0, "no_window": 0}
for address, g in daily.groupby("address"):
    g = g.set_index("date")
    grid = pd.date_range(g.index.min(), LAST_MARK.floor("D"), freq="D")
    observed = pd.Series(True, index=g.index).reindex(grid, fill_value=False)   # a real mark on this day
    full = g.reindex(grid).ffill()
    p = full["share_price"].where(full["share_price"] > 0)
    tvl = full["total_assets"]
    r = np.log(p).diff()
    born = g.index.min()
    mark_days = observed[observed].index
    for t in decisions:
        if t not in full.index:
            continue
        t1 = t - pd.Timedelta(days=1)
        # Eligibility on OBSERVED information: a real mark within MAX_STALE_DAYS of T-1, and the
        # TVL from that mark. A vault whose last mark is older is not a live candidate.
        recent = mark_days[(mark_days <= t1) & (mark_days > t1 - pd.Timedelta(days=MAX_STALE_DAYS))]
        if len(recent) == 0:
            dropped["no_recent_mark"] += 1
            continue
        if not (g.loc[recent[-1], "total_assets"] >= MIN_TVL_USD):
            dropped["tvl"] += 1
            continue
        age = int((t - born).days)
        row = {"address": address, "date": t, "age_days": age, "young": age < YOUNG_DAYS,
               "days_since_mark": int((t1 - recent[-1]).days)}
        scored_any = False
        for w in WINDOWS:
            win = r[(r.index > t1 - pd.Timedelta(days=w)) & (r.index <= t1)].dropna().to_numpy()
            if len(win) < w or int((np.abs(win) > 0).sum()) < MIN_FRESH:
                for k in TRIMS:
                    row[f"ret{w}_k{k}"] = np.nan; row[f"sharpe{w}_k{k}"] = np.nan
                row[f"sortino{w}"] = np.nan; row[f"vol{w}"] = np.nan
                row[f"fresh{w}"] = int((np.abs(win) > 0).sum()) if len(win) else 0
                continue
            scored_any = True
            s = trailing_scores(win)
            for key, value in s.items():
                row[f"{key.split('_')[0]}{w}_{key.split('_')[1]}" if "_" in key else f"{key}{w}"] = value
            row[f"fresh{w}"] = int((np.abs(win) > 0).sum())
        if not scored_any:
            dropped["no_window"] += 1
            continue
        t_end = t + pd.Timedelta(days=FORWARD_DAYS)
        fwd = r[(r.index > t) & (r.index <= t_end)].dropna().to_numpy()
        fwd_marks = mark_days[(mark_days > t) & (mark_days <= t_end)]
        # The forward window must be OBSERVED, not manufactured: enough real marks and one near
        # its end. Otherwise a vault that stopped reporting scores as perfectly calm.
        if len(fwd) < FORWARD_DAYS or len(fwd_marks) < MIN_FORWARD_MARKS or                 (len(fwd_marks) and (t_end - fwd_marks[-1]).days > MAX_STALE_DAYS) or not len(fwd_marks):
            dropped["forward_marks"] += 1
            continue
        row["fwd_marks"] = int(len(fwd_marks))
        row.update(forward_targets(fwd, None))
        rows.append(row)
panel = pd.DataFrame(rows)
print("candidate-dates dropped:", dropped)
print(f"forward-window observed marks: median {panel['fwd_marks'].median():.0f} of {FORWARD_DAYS} days, "
      f"5th percentile {panel['fwd_marks'].quantile(0.05):.0f}; days since last mark at T-1: mean {panel['days_since_mark'].mean():.2f}")
panel["name"] = panel["address"].map(names)
print(f"panel: {len(panel):,} rows, {panel['address'].nunique()} vaults, {panel['date'].nunique()} decisions "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}; young (< {YOUNG_DAYS} d) share of rows "
      f"{panel['young'].mean():.1%}")
SIGNALS = []
for w in WINDOWS:
    for k in TRIMS:
        SIGNALS.append({"name": f"ret{w}_k{k}", "direction": "high", "window": w, "k": k, "family": "return"})
    for k in TRIMS:
        SIGNALS.append({"name": f"sharpe{w}_k{k}", "direction": "high", "window": w, "k": k, "family": "sharpe"})
    SIGNALS.append({"name": f"sortino{w}", "direction": "high", "window": w, "k": None, "family": "sortino"})
    SIGNALS.append({"name": f"vol{w}", "direction": "low", "window": w, "k": None, "family": "vol"})
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
TARGETS = ["fwd_sharpe", "fwd_return", "fwd_vol", "fwd_log_max_dd"]
#: positive = the signal's good end had the better outcome: higher Sharpe/return, LOWER vol, shallower (larger) max DD
TARGET_SIGN = {"fwd_sharpe": 1.0, "fwd_return": 1.0, "fwd_vol": -1.0, "fwd_log_max_dd": 1.0}
coverage = pd.DataFrame({s: np.isfinite(panel[s]).mean() for s in SIGNAL_NAMES}, index=["finite_share"]).T
display(coverage.round(3).T)
display(panel[TARGETS].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(4))


candidate-dates dropped: {'no_recent_mark': 7705, 'tvl': 13244, 'forward_marks': 18, 'no_window': 2022}
forward-window observed marks: median 30 of 30 days, 5th percentile 30; days since last mark at T-1: mean 0.00
panel: 17,197 rows, 364 vaults, 70 decisions 2026-04-01 to 2026-08-17; young (< 360 d) share of rows 79.9%


,ret45_k0,ret45_k3,ret45_k5,ret45_k10,sharpe45_k0,sharpe45_k3,sharpe45_k5,sharpe45_k10,sortino45,vol45,ret90_k0,ret90_k3,ret90_k5,ret90_k10,sharpe90_k0,sharpe90_k3,sharpe90_k5,sharpe90_k10,sortino90,vol90,ret180_k0,ret180_k3,ret180_k5,ret180_k10,sharpe180_k0,sharpe180_k3,sharpe180_k5,sharpe180_k10,sortino180,vol180
finite_share,0.941,0.941,0.941,0.941,0.941,0.941,0.941,0.94,0.937,0.941,0.861,0.861,0.861,0.861,0.861,0.861,0.861,0.861,0.859,0.861,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645,0.645


,fwd_sharpe,fwd_return,fwd_vol,fwd_log_max_dd
count,16460.0000,17197.0000,17197.0000,17197.0000
mean,0.6474,-0.0646,0.6454,-0.2136
std,9.3991,0.4784,1.0336,0.4866
min,-14.2720,-13.1463,0.0000,-13.1463
5%,-6.2449,-0.6322,0.0000,-0.9061
25%,-3.0534,-0.0577,0.0982,-0.2033
50%,0.0855,0.0000,0.2918,-0.0691
75%,3.1582,0.0554,0.7948,-0.0164
95%,6.8106,0.3160,2.4002,0.0000
max,234.4167,1.9923,16.4087,0.0000


## Part 2. One shared bootstrap, every hypothesis

Per date and signal, the complete-case Spearman against each target; equal-weight mean over
dates. The bootstrap resamples 15-decision circular date blocks and vault clusters together, and
every hypothesis - every signal, every target, and every trimmed-minus-raw difference - is a
function of the same draws.


In [3]:
def per_date_blocks(frame: pd.DataFrame) -> dict:
    """{date: {'vault': array, 'values': (n, S+T) float array}} with NaN where a signal is missing."""
    out = {}
    cols = SIGNAL_NAMES + TARGETS
    for date, g in frame.groupby("date"):
        out[pd.Timestamp(date)] = {"vault": g["address"].to_numpy(), "values": g[cols].to_numpy(dtype=float)}
    return out


def date_statistics(values: np.ndarray) -> np.ndarray:
    """(S, T) signed Spearman on the complete cases of each (signal, target) pair; NaN if fewer than
    MIN_CANDIDATES rows or a constant column."""
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    out = np.full((S, T), np.nan)
    targets = values[:, S:]
    for i, name in enumerate(SIGNAL_NAMES):
        x = values[:, i]
        for j, target in enumerate(TARGETS):
            y = targets[:, j]
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < MIN_CANDIDATES:
                continue
            xr, yr = rankdata(x[ok]), rankdata(y[ok])
            if np.ptp(xr) == 0 or np.ptp(yr) == 0:
                continue
            xc, yc = xr - xr.mean(), yr - yr.mean()
            rho = float((xc * yc).sum() / math.sqrt((xc ** 2).sum() * (yc ** 2).sum()))
            out[i, j] = SIGNAL_SIGN[name] * TARGET_SIGN[target] * rho
    return out


def mean_over_dates(blocks: dict, dates: list, counts: dict | None = None) -> tuple:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    total, n = np.zeros((S, T)), np.zeros((S, T))
    for date in dates:
        block = blocks.get(date)
        if block is None:
            continue
        values = block["values"]
        if counts is not None:
            repeats = np.array([counts.get(v, 0) for v in block["vault"]], dtype=int)
            if repeats.sum() < MIN_CANDIDATES:
                continue
            values = np.repeat(values, repeats, axis=0)
        stats = date_statistics(values)
        finite = np.isfinite(stats)
        total[finite] += stats[finite]
        n[finite] += 1
    with np.errstate(invalid="ignore"):
        return np.where(n > 0, total / np.maximum(n, 1), np.nan), n


def bootstrap(frame: pd.DataFrame, draws: int = DRAWS, seed: int = SEED, verbose: bool = True) -> dict:
    blocks = per_date_blocks(frame)
    dates = sorted(blocks)
    vaults = sorted(frame["address"].unique())
    observed, n_dates = mean_over_dates(blocks, dates)
    rng = np.random.default_rng(seed)
    n_blocks = int(math.ceil(len(dates) / DATE_BLOCK))
    reps = np.full((draws,) + observed.shape, np.nan)
    for d in range(draws):
        starts = rng.integers(0, len(dates), size=n_blocks)
        index = np.concatenate([(np.arange(s, s + DATE_BLOCK) % len(dates)) for s in starts])[:len(dates)]
        drawn_vaults = rng.choice(len(vaults), size=len(vaults), replace=True)
        counts = {}
        for v in drawn_vaults:
            counts[vaults[v]] = counts.get(vaults[v], 0) + 1
        reps[d], _ = mean_over_dates(blocks, [dates[i] for i in index], counts)
        if verbose and (d + 1) % 100 == 0:
            print(f"  draw {d + 1}/{draws}")
    return {"observed": observed, "draws": reps, "n_dates": n_dates, "dates": dates, "rows": len(frame)}


def simultaneous_lower(observed: np.ndarray, reps: np.ndarray, level: float = LEVEL) -> dict:
    """One-sided simultaneous lower bounds by studentised max-T over the finite family, on
    complete-family draws only; add-one p for theta <= 0."""
    se = np.nanstd(reps, axis=0, ddof=1)
    member = np.isfinite(observed) & np.isfinite(se) & (se > 0)
    stud = (reps - observed[None, :]) / np.where(se > 0, se, np.nan)[None, :]
    complete = np.isfinite(stud[:, member]).all(axis=1) if member.any() else np.zeros(len(reps), dtype=bool)
    per_draw_max = stud[complete][:, member].max(axis=1) if complete.any() else np.array([])
    critical = float(np.percentile(per_draw_max, level * 100.0)) if len(per_draw_max) >= 100 else float("nan")
    lower = np.where(member, observed - critical * se, np.nan)
    unadjusted = np.nanpercentile(reps, (1 - level) * 100.0, axis=0)
    centred = reps - observed[None, :]
    p = (1.0 + (centred >= observed[None, :]).sum(axis=0)) / (len(reps) + 1.0)
    return {"se": se, "critical": critical, "lower": lower, "lower_unadjusted": unadjusted, "p": p,
            "family_size": int(member.sum()), "complete_draws": int(len(per_draw_max))}


def screen(frame: pd.DataFrame, label: str, verbose: bool = True) -> dict:
    print(f"{label}: {len(frame):,} rows, {frame['date'].nunique()} decisions, {frame['address'].nunique()} vaults")
    boot = bootstrap(frame, verbose=verbose)
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    j = TARGETS.index("fwd_sharpe")
    fam = simultaneous_lower(boot["observed"][:, j], boot["draws"][:, :, j])
    rows = []
    for i, s in enumerate(SIGNALS):
        row = {"signal": s["name"], "family": s["family"], "window": s["window"], "k": s["k"],
               "dates": int(boot["n_dates"][i, j]), "evaluated": bool(boot["n_dates"][i, j] >= MIN_DATES and fam["se"][i] > 0)}
        for t_idx, target in enumerate(TARGETS):
            row[f"rho_{target}"] = boot["observed"][i, t_idx]
        row["se_sharpe"] = fam["se"][i]
        row["lo_sharpe_simultaneous"] = fam["lower"][i]
        row["lo_sharpe_unadjusted"] = fam["lower_unadjusted"][i]
        row["p_sharpe"] = fam["p"][i]
        rows.append(row)
    table = pd.DataFrame(rows).set_index("signal")
    # Paired trimmed-minus-raw differences on the primary target, on the same draws. The
    # per-comparison interval and add-one p are DESCRIPTIVE; the 18 comparisons are also one
    # family, so a simultaneous max-T lower bound over them is reported as the controlled figure.
    diffs, d_obs, d_reps = [], [], []
    for w in WINDOWS:
        for fam_name in ("ret", "sharpe"):
            base = SIGNAL_NAMES.index(f"{fam_name}{w}_k0")
            for k in TRIMS[1:]:
                idx = SIGNAL_NAMES.index(f"{fam_name}{w}_k{k}")
                obs = boot["observed"][idx, j] - boot["observed"][base, j]
                rep = boot["draws"][:, idx, j] - boot["draws"][:, base, j]
                d_obs.append(obs); d_reps.append(rep)
                fin = rep[np.isfinite(rep)]
                centred = fin - obs
                n = len(fin)
                p_hi = (1.0 + (centred >= obs).sum()) / (n + 1.0)
                p_lo = (1.0 + (centred <= obs).sum()) / (n + 1.0)
                diffs.append({"family": fam_name, "window": w, "k": k, "trimmed_rho": boot["observed"][idx, j],
                              "raw_rho": boot["observed"][base, j], "difference": obs,
                              "ci_lo": float(np.percentile(fin, 2.5)) if n >= 100 else np.nan,
                              "ci_hi": float(np.percentile(fin, 97.5)) if n >= 100 else np.nan,
                              "p_two_sided_add_one": float(min(1.0, 2 * min(p_hi, p_lo))) if n >= 100 else np.nan,
                              "draws": int(n)})
    paired = pd.DataFrame(diffs)
    pfam = simultaneous_lower(np.array(d_obs), np.column_stack(d_reps))
    paired["lo_simultaneous_18"] = pfam["lower"]
    paired["se"] = pfam["se"]
    return {"label": label, "table": table, "paired": paired, "critical": fam["critical"],
            "family_size": fam["family_size"], "complete_draws": fam["complete_draws"], "boot": boot,
            "paired_critical": pfam["critical"], "paired_family_size": pfam["family_size"]}


full = screen(panel, "all candidates")
print(f"\nprimary family: {full['family_size']} signals, critical {full['critical']:.4f} on {full['complete_draws']} complete draws")
display(full["table"][["family", "window", "k", "dates", "evaluated", "rho_fwd_sharpe", "se_sharpe", "lo_sharpe_simultaneous",
                       "lo_sharpe_unadjusted", "p_sharpe", "rho_fwd_return", "rho_fwd_vol", "rho_fwd_log_max_dd"]].round(4))
print(f"\nPAIRED trimmed - raw on forward Sharpe (95% percentile interval on shared draws; simultaneous lower bound over "
      f"the {full['paired_family_size']} paired comparisons, critical {full['paired_critical']:.4f}):")
display(full["paired"].round(4))


all candidates: 17,197 rows, 70 decisions, 364 vaults


  draw 100/500


  draw 200/500


  draw 300/500


  draw 400/500


  draw 500/500

primary family: 30 signals, critical 2.5335 on 500 complete draws


,family,window,k,dates,evaluated,rho_fwd_sharpe,se_sharpe,lo_sharpe_simultaneous,lo_sharpe_unadjusted,p_sharpe,rho_fwd_return,rho_fwd_vol,rho_fwd_log_max_dd
signal,,,,,,,,,,,,,
ret45_k0,return,45,0.0,70,True,-0.0045,0.0536,-0.1402,-0.0947,0.5389,-0.0498,0.0738,0.0480
ret45_k3,return,45,3.0,70,True,0.0420,0.0462,-0.0750,-0.0328,0.1856,-0.0340,0.4819,0.4203
ret45_k5,return,45,5.0,70,True,0.0680,0.0444,-0.0444,-0.0016,0.0659,-0.0111,0.5972,0.5329
ret45_k10,return,45,10.0,70,True,0.0952,0.0436,-0.0151,0.0226,0.0160,0.0197,0.6965,0.6336
sharpe45_k0,sharpe,45,0.0,70,True,0.0494,0.0553,-0.0907,-0.0433,0.1776,-0.0352,0.1268,0.1031
sharpe45_k3,sharpe,45,3.0,70,True,0.0440,0.0561,-0.0980,-0.0457,0.2136,-0.0388,0.1004,0.0774
sharpe45_k5,sharpe,45,5.0,70,True,0.0483,0.0551,-0.0913,-0.0430,0.2036,-0.0414,0.1292,0.1082
sharpe45_k10,sharpe,45,10.0,70,True,0.0502,0.0456,-0.0653,-0.0201,0.1437,-0.0367,0.2259,0.2146
sortino45,sortino,45,NaN,70,True,0.0425,0.0547,-0.0962,-0.0478,0.2375,-0.0326,0.1335,0.1093



PAIRED trimmed - raw on forward Sharpe (95% percentile interval on shared draws; simultaneous lower bound over the 18 paired comparisons, critical 2.3944):


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_18,se
0,ret,45,3,0.0420,-0.0045,0.0465,-0.0155,0.1119,0.1796,500,-0.0335,0.0334
1,ret,45,5,0.0680,-0.0045,0.0725,-0.0131,0.1632,0.1158,500,-0.0359,0.0453
2,ret,45,10,0.0952,-0.0045,0.0997,-0.0067,0.2150,0.0958,500,-0.0343,0.0560
3,sharpe,45,3,0.0440,0.0494,-0.0054,-0.0280,0.0179,0.6946,500,-0.0349,0.0123
4,sharpe,45,5,0.0483,0.0494,-0.0011,-0.0420,0.0361,0.9780,500,-0.0499,0.0204
5,sharpe,45,10,0.0502,0.0494,0.0009,-0.0675,0.0731,0.9102,500,-0.0880,0.0371
6,ret,90,3,0.0463,-0.0195,0.0657,0.0018,0.1274,0.0519,500,-0.0139,0.0333
7,ret,90,5,0.0646,-0.0195,0.0840,0.0001,0.1615,0.0439,500,-0.0166,0.0420
8,ret,90,10,0.0865,-0.0195,0.1059,0.0040,0.2057,0.0399,500,-0.0158,0.0509
9,sharpe,90,3,0.0229,0.0318,-0.0089,-0.0376,0.0172,0.5629,500,-0.0420,0.0138


### Reachability: can this screen produce a positive simultaneous bound at all?

Standing rule 9. A noisy foresight oracle - the forward Sharpe itself plus 5% noise - is added
to the family and run through the identical panel, bootstrap and max-T. If it does not clear
the family-wise lower bound, the all-fail result above is a property of the machinery, not of
the signals. Diagnostic only; nothing here is reported as a finding.


In [4]:
rng = np.random.default_rng(SEED + 1)
oracle_panel = panel.copy()
oracle_panel["oracle_fwd_sharpe"] = oracle_panel["fwd_sharpe"] + rng.normal(0.0, 0.05 * float(oracle_panel["fwd_sharpe"].std()), len(oracle_panel))
_saved = (SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN)
SIGNALS = list(SIGNALS) + [{"name": "oracle_fwd_sharpe", "direction": "high", "window": None, "k": None, "family": "oracle"}]
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
try:
    oracle_res = screen(oracle_panel, "oracle reachability (31-signal family)", verbose=False)
finally:
    SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN = _saved
orow = oracle_res["table"].loc["oracle_fwd_sharpe"]
print(f"oracle: rho {orow['rho_fwd_sharpe']:.4f}, simultaneous lower bound {orow['lo_sharpe_simultaneous']:.4f} over "
      f"{oracle_res['family_size']} signals (critical {oracle_res['critical']:.4f}), finite target rows "
      f"{int(np.isfinite(panel['fwd_sharpe']).sum())} of {len(panel)}")
assert orow["lo_sharpe_simultaneous"] > 0, "the screen cannot produce a positive simultaneous bound even for a foresight oracle"
print("reachable: a foresight signal clears the family-wise bound; the thirty real signals fail it on their merits")


oracle reachability (31-signal family): 17,197 rows, 70 decisions, 364 vaults


oracle: rho 0.9908, simultaneous lower bound 0.9886 over 31 signals (critical 2.5335), finite target rows 16460 of 17197
reachable: a foresight signal clears the family-wise bound; the thirty real signals fail it on their merits


## Part 3. The young cohort and the old

Vaults under 360 days old are unscorable by the incumbent's CAGR leg; they are where a
shorter-window ranker would matter most. The screen is repeated on the young rows and on the
rest, separately, so the two are not averaged into each other.


In [5]:
young = screen(panel[panel["young"]], "young (< 360 days)", verbose=False)
old = screen(panel[~panel["young"]], "old (>= 360 days)", verbose=False)
COLS = ["window", "k", "dates", "evaluated", "rho_fwd_sharpe", "lo_sharpe_simultaneous", "p_sharpe", "rho_fwd_return", "rho_fwd_vol"]
print(f"note: the young and old screens are separate bootstraps on separate samples; their paired-difference intervals are descriptive")
for res in (young, old):
    print(f"\n{res['label']}: family {res['family_size']}, critical {res['critical']:.4f}, complete draws {res['complete_draws']}")
    display(res["table"][COLS].round(4))
    print("paired trimmed - raw on forward Sharpe:")
    display(res["paired"].round(4))


young (< 360 days): 13,742 rows, 70 decisions, 357 vaults


old (>= 360 days): 3,455 rows, 40 decisions, 114 vaults


note: the young and old screens are separate bootstraps on separate samples; their paired-difference intervals are descriptive

young (< 360 days): family 30, critical 2.5328, complete draws 500


,window,k,dates,evaluated,rho_fwd_sharpe,lo_sharpe_simultaneous,p_sharpe,rho_fwd_return,rho_fwd_vol
signal,,,,,,,,,
ret45_k0,45,0.0,70,True,-0.0180,-0.1707,0.6447,-0.0562,0.0625
ret45_k3,45,3.0,70,True,0.0375,-0.1021,0.2455,-0.0244,0.4692
ret45_k5,45,5.0,70,True,0.0681,-0.0640,0.0998,0.0039,0.5880
ret45_k10,45,10.0,70,True,0.1002,-0.0272,0.0240,0.0385,0.6883
sharpe45_k0,45,0.0,70,True,0.0418,-0.1170,0.2794,-0.0373,0.1185
sharpe45_k3,45,3.0,70,True,0.0372,-0.1209,0.3054,-0.0417,0.0928
sharpe45_k5,45,5.0,70,True,0.0421,-0.1118,0.2615,-0.0452,0.1223
sharpe45_k10,45,10.0,70,True,0.0418,-0.0953,0.2315,-0.0440,0.2214
sortino45,45,NaN,70,True,0.0300,-0.1295,0.3553,-0.0356,0.1227


paired trimmed - raw on forward Sharpe:


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_18,se
0,ret,45,3,0.0375,-0.0180,0.0555,-0.0126,0.1204,0.0958,500,-0.0322,0.0341
1,ret,45,5,0.0681,-0.0180,0.0861,-0.0058,0.1706,0.0519,500,-0.0312,0.0456
2,ret,45,10,0.1002,-0.0180,0.1181,-0.0030,0.2213,0.0279,500,-0.0302,0.0577
3,sharpe,45,3,0.0372,0.0418,-0.0046,-0.0319,0.0240,0.7705,500,-0.0412,0.0142
4,sharpe,45,5,0.0421,0.0418,0.0003,-0.0424,0.0500,0.9541,500,-0.0585,0.0229
5,sharpe,45,10,0.0418,0.0418,0.0000,-0.0838,0.0799,0.9541,500,-0.1053,0.0410
6,ret,90,3,0.0398,-0.0383,0.0782,0.0100,0.1387,0.0080,500,-0.0079,0.0335
7,ret,90,5,0.0621,-0.0383,0.1005,0.0131,0.1778,0.0080,500,-0.0119,0.0437
8,ret,90,10,0.0884,-0.0383,0.1267,0.0160,0.2272,0.0160,500,-0.0147,0.0550
9,sharpe,90,3,0.0152,0.0216,-0.0064,-0.0376,0.0273,0.7465,500,-0.0497,0.0168



old (>= 360 days): family 30, critical 2.7077, complete draws 500


,window,k,dates,evaluated,rho_fwd_sharpe,lo_sharpe_simultaneous,p_sharpe,rho_fwd_return,rho_fwd_vol
signal,,,,,,,,,
ret45_k0,45,0.0,40,True,0.0196,-0.1594,0.3713,-0.0419,0.1754
ret45_k3,45,3.0,40,True,0.0275,-0.1765,0.3613,-0.0920,0.5594
ret45_k5,45,5.0,40,True,0.0414,-0.1623,0.3114,-0.0860,0.6534
ret45_k10,45,10.0,40,True,0.0517,-0.1559,0.2495,-0.0698,0.7526
sharpe45_k0,45,0.0,40,True,0.0421,-0.1712,0.2754,-0.0545,0.2004
sharpe45_k3,45,3.0,40,True,0.0462,-0.1769,0.2794,-0.0449,0.1621
sharpe45_k5,45,5.0,40,True,0.0609,-0.1522,0.1976,-0.0319,0.1798
sharpe45_k10,45,10.0,40,True,0.0825,-0.1052,0.1058,-0.0037,0.2369
sortino45,45,NaN,40,True,0.0428,-0.1720,0.2834,-0.0532,0.2137


paired trimmed - raw on forward Sharpe:


,family,window,k,trimmed_rho,raw_rho,difference,ci_lo,ci_hi,p_two_sided_add_one,draws,lo_simultaneous_18,se
0,ret,45,3,0.0275,0.0196,0.0079,-0.1003,0.1195,0.8822,500,-0.1327,0.0528
1,ret,45,5,0.0414,0.0196,0.0217,-0.1047,0.1574,0.7305,500,-0.1470,0.0633
2,ret,45,10,0.0517,0.0196,0.0320,-0.1160,0.1875,0.6786,500,-0.1748,0.0776
3,sharpe,45,3,0.0462,0.0421,0.0041,-0.0424,0.0430,0.8263,500,-0.0553,0.0223
4,sharpe,45,5,0.0609,0.0421,0.0188,-0.0387,0.0709,0.4271,500,-0.0536,0.0272
5,sharpe,45,10,0.0825,0.0421,0.0405,-0.0603,0.1589,0.4192,500,-0.1086,0.0559
6,ret,90,3,0.0133,-0.0112,0.0245,-0.1003,0.1881,0.7385,500,-0.1743,0.0746
7,ret,90,5,0.0235,-0.0112,0.0347,-0.1182,0.2157,0.7146,500,-0.1985,0.0875
8,ret,90,10,0.0378,-0.0112,0.0490,-0.1338,0.2477,0.6427,500,-0.2168,0.0997
9,sharpe,90,3,0.0209,0.0239,-0.0030,-0.0564,0.0396,0.8104,500,-0.0675,0.0242


## Part 4. Stratwise and the post-July cohort: a current-snapshot comparison

Stratwise Multi-Asset Public has 63 days of history at this archive. It cannot appear in the
screen: no decision date gives it a 45-day trailing window AND a complete 30-day forward window.
What can be shown is how raw and trimmed trailing scores rank it and the other young vaults on
the LATEST date at which each window can be computed, against every vault scorable on that
date. This is description; nothing here is evidence about forward behaviour.


In [6]:
snapshot_rows = []
snap_dates = {}
SNAP_DAY = LAST_MARK.floor("D") - pd.Timedelta(days=1)   # the last COMPLETED UTC day, not the partial one
for w in WINDOWS:
    t = SNAP_DAY
    snap_dates[w] = t
    for address, g in daily.groupby("address"):
        g = g.set_index("date")
        if g.index.min() > t:
            continue
        grid = pd.date_range(g.index.min(), t, freq="D")
        full_ = g.reindex(grid).ffill()
        p = full_["share_price"].where(full_["share_price"] > 0)
        recent_marks = g.index[(g.index <= t) & (g.index > t - pd.Timedelta(days=MAX_STALE_DAYS))]
        if len(recent_marks) == 0 or not (g.loc[recent_marks[-1], "total_assets"] >= MIN_TVL_USD):
            continue
        r = np.log(p).diff()
        win = r[(r.index > t - pd.Timedelta(days=w)) & (r.index <= t)].dropna().to_numpy()
        if len(win) < w or int((np.abs(win) > 0).sum()) < MIN_FRESH:
            continue
        s = trailing_scores(win)
        snapshot_rows.append({"window": w, "address": address, "name": names.get(address, ""),
                              "age_days": int((t - g.index.min()).days), "tvl": float(full_["total_assets"].iloc[-1]),
                              "fresh": int((np.abs(win) > 0).sum()), **s})
snapshot = pd.DataFrame(snapshot_rows)
for w in WINDOWS:
    sub = snapshot[snapshot["window"] == w].copy()
    for col in ["ret_k0", "ret_k5", "sharpe_k0", "sharpe_k5"]:
        sub[f"rank_{col}"] = sub[col].rank(ascending=False, method="min")
    sub = sub.sort_values("sharpe_k5", ascending=False)
    print(f"\nwindow {w} rows at {snap_dates[w].date()}: {len(sub)} scorable vaults; post-July (age < 75 d): {(sub['age_days'] < 75).sum()}")
    show = sub[(sub["age_days"] < 120) | (sub["address"] == STRATWISE)].head(25)
    display(show[["name", "age_days", "tvl", "fresh", "ret_k0", "ret_k5", "sharpe_k0", "sharpe_k5", "sortino", "vol",
                  "rank_ret_k0", "rank_ret_k5", "rank_sharpe_k0", "rank_sharpe_k5"]].round(3).set_index("name"))
    if (sub["address"] == STRATWISE).any():
        sw = sub[sub["address"] == STRATWISE].iloc[0]
        print(f"  Stratwise at window {w}: raw return {sw['ret_k0']:+.3f} (rank {int(sw['rank_ret_k0'])}/{len(sub)}), trimmed k=5 "
              f"{sw['ret_k5']:+.3f} (rank {int(sw['rank_ret_k5'])}); raw Sharpe {sw['sharpe_k0']:.2f} (rank {int(sw['rank_sharpe_k0'])}), "
              f"trimmed k=5 {sw['sharpe_k5']:.2f} (rank {int(sw['rank_sharpe_k5'])})")
    else:
        print(f"  Stratwise has no {w}-row window at {snap_dates[w].date()}")
# How much trimming moves the cross-sectional ordering at all, per window: rank correlation raw vs trimmed.
agreement = []
for w in WINDOWS:
    sub = snapshot[snapshot["window"] == w]
    for fam_name in ("ret", "sharpe"):
        for k in TRIMS[1:]:
            agreement.append({"window": w, "family": fam_name, "k": k,
                              "spearman_raw_vs_trimmed": float(sub[f"{fam_name}_k0"].corr(sub[f"{fam_name}_k{k}"], method="spearman")),
                              "vaults": len(sub)})
display(pd.DataFrame(agreement).round(3))



window 45 rows at 2026-09-15: 221 scorable vaults; post-July (age < 75 d): 3


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,
[Bee] Line,54,27965.101,43,2.371,1.075,8.144,4.923,16.416,0.291,47.0,5.0,8.0,10.0
Stratwise Multi-Asset Public,61,186026.588,45,0.309,0.074,6.436,2.401,11.423,0.048,119.0,20.0,13.0,16.0
DOUBLETOP Vault,88,5184594.726,45,1.126,0.004,4.323,0.034,11.288,0.260,76.0,28.0,42.0,29.0
Kairos Fi,110,42058.787,36,2.068,-1.178,3.048,-2.853,5.601,0.679,48.0,135.0,78.0,81.0
korea wins again,82,110682.039,45,-2.479,-4.683,-2.801,-6.425,-3.273,0.885,205.0,193.0,193.0,169.0
NEET WORLD ORDER,112,329624.450,45,-1.949,-6.791,-1.510,-7.973,-2.054,1.291,202.0,208.0,175.0,206.0
Nova Quant,98,29215.787,45,-0.743,-1.063,-5.253,-9.108,-5.547,0.141,186.0,125.0,214.0,213.0
Genesis Algo 88 | BABYSATOSHI,47,7927.863,31,-38.129,-52.014,-7.213,-12.796,-7.491,5.286,221.0,221.0,221.0,221.0


  Stratwise at window 45: raw return +0.309 (rank 119/221), trimmed k=5 +0.074 (rank 20); raw Sharpe 6.44 (rank 13), trimmed k=5 2.40 (rank 16)

window 90 rows at 2026-09-15: 227 scorable vaults; post-July (age < 75 d): 0


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,
Kairos Fi,110,42058.787,43,0.883,-0.740,1.817,-2.411,3.307,0.486,38.0,119.0,50.0,89.0
NEET WORLD ORDER,112,329624.450,74,-0.923,-3.404,-0.879,-4.164,-1.143,1.050,198.0,195.0,173.0,173.0
Nova Quant,98,29215.787,90,-0.164,-0.700,-0.767,-4.353,-1.080,0.213,169.0,116.0,167.0,178.0


  Stratwise has no 90-row window at 2026-09-15

window 180 rows at 2026-09-15: 198 scorable vaults; post-July (age < 75 d): 0


,age_days,tvl,fresh,ret_k0,ret_k5,sharpe_k0,sharpe_k5,sortino,vol,rank_ret_k0,rank_ret_k5,rank_sharpe_k0,rank_sharpe_k5
name,,,,,,,,,,,,,


  Stratwise has no 180-row window at 2026-09-15


,window,family,k,spearman_raw_vs_trimmed,vaults
0,45,ret,3,0.564,221
1,45,ret,5,0.283,221
2,45,ret,10,0.014,221
3,45,sharpe,3,0.953,221
4,45,sharpe,5,0.898,221
5,45,sharpe,10,0.653,221
6,90,ret,3,0.478,227
7,90,ret,5,0.304,227
8,90,ret,10,0.122,227
9,90,sharpe,3,0.905,227


## Part 5. Manifest


In [7]:
def table_records(res):
    return {"table": res["table"].round(6).to_dict(orient="index"), "paired": res["paired"].round(6).to_dict(orient="records"),
            "critical": res["critical"], "family_size": res["family_size"], "complete_draws": res["complete_draws"],
            "rows": int(res["boot"]["rows"]), "decisions": int(len(res["boot"]["dates"]))}

manifest = {
    "verdict": "DIAGNOSTIC - a vault-level screen, not a result",
    "provenance": {**PROVENANCE, "last_mark": str(LAST_MARK)},
    "constants": {"windows": list(WINDOWS), "trims": list(TRIMS), "forward_days": FORWARD_DAYS, "post_break_start": str(POST_BREAK_START.date()),
                  "min_tvl_usd": MIN_TVL_USD, "min_fresh": MIN_FRESH, "min_candidates": MIN_CANDIDATES, "min_dates": MIN_DATES,
                  "young_days": YOUNG_DAYS, "draws": DRAWS, "date_block": DATE_BLOCK, "seed": SEED},
    "panel": {"rows": int(len(panel)), "vaults": int(panel["address"].nunique()), "decisions": int(panel["date"].nunique()),
              "first": str(panel["date"].min().date()), "last": str(panel["date"].max().date()), "young_share": float(panel["young"].mean())},
    "coverage": coverage["finite_share"].round(6).to_dict(),
    "screens": {"all": table_records(full), "young": table_records(young), "old": table_records(old)},
    "snapshot_dates": {str(w): str(t.date()) for w, t in snap_dates.items()},
    "snapshot_agreement": pd.DataFrame(agreement).round(6).to_dict(orient="records"),
    "stratwise": {str(w): (snapshot[(snapshot["window"] == w) & (snapshot["address"] == STRATWISE)].drop(columns=["address"]).round(6).to_dict(orient="records"))
                  for w in WINDOWS},
    "stratwise_age_days": int((SNAP_DAY - daily[daily["address"] == STRATWISE]["date"].min()).days),
    "dropped": dropped,
    "forward_marks_median": float(panel["fwd_marks"].median()), "forward_marks_p05": float(panel["fwd_marks"].quantile(0.05)),
    "oracle": {"rho": float(orow["rho_fwd_sharpe"]), "lo_simultaneous": float(orow["lo_sharpe_simultaneous"]),
               "family_size": oracle_res["family_size"], "critical": oracle_res["critical"]},
    "paired_family": {k: {"critical": SC_["paired_critical"], "size": SC_["paired_family_size"]} for k, SC_ in (("all", full), ("young", young), ("old", old))},
}
Path("_build/manifest_38.json").write_text(json.dumps(manifest, indent=1, default=str))
print("wrote _build/manifest_38.json")


wrote _build/manifest_38.json
